# Lab Tùy chọn: Chuẩn hóa đặc trưng và Tốc độ học (Nhiều biến)

## Mục tiêu
Trong lab này, bạn sẽ:
- Sử dụng các hàm xử lý nhiều biến đã xây dựng ở lab trước
- chạy Gradient Descent trên một tập dữ liệu có nhiều đặc trưng
- khám phá ảnh hưởng của *tốc độ học alpha* đến gradient descent
- cải thiện hiệu năng của gradient descent bằng cách *chuẩn hóa đặc trưng* sử dụng chuẩn hóa z-score

## Công cụ
Bạn sẽ sử dụng các hàm đã xây dựng ở lab trước cũng như matplotlib và NumPy. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lab_utils_multi import  load_house_data, run_gradient_descent 
from lab_utils_multi import  norm_plot, plt_equal_scale, plot_cost_i_w
from lab_utils_common import dlc
np.set_printoptions(precision=2)
plt.style.use('./deeplearning.mplstyle')

## Ký hiệu

|Ký hiệu <br />  chung  | Mô tả| Python (nếu có) |
|: ------------|: ------------------------------------------------------------||
| $a$ | vô hướng, không in đậm                                                      ||
| $\mathbf{a}$ | vector, in đậm                                                 ||
| $\mathbf{A}$ | ma trận, chữ hoa in đậm                                         ||
| **Hồi quy** |         |    |     |
|  $\mathbf{X}$ | ma trận các ví dụ huấn luyện                  | `X_train` |   
|  $\mathbf{y}$  | nhãn (target) của các ví dụ huấn luyện                | `y_train` 
|  $\mathbf{x}^{(i)}$, $y^{(i)}$ | Ví dụ huấn luyện thứ $i$ | `X[i]`, `y[i]`|
| m | số lượng ví dụ huấn luyện | `m`|
| n | số lượng đặc trưng trong mỗi ví dụ | `n`|
|  $\mathbf{w}$  |  tham số: trọng số,                       | `w`    |
|  $b$           |  tham số: độ chệch (bias)                                           | `b`    |     
| $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ | Kết quả đánh giá mô hình tại $\mathbf{x}^{(i)}$ với tham số $\mathbf{w},b$: $f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)}+b$  | `f_wb` | 
|$\frac{\partial J(\mathbf{w},b)}{\partial w_j}$| đạo hàm riêng của hàm chi phí theo tham số $w_j$ |`dj_dw[j]`| 
|$\frac{\partial J(\mathbf{w},b)}{\partial b}$| đạo hàm riêng của hàm chi phí theo tham số $b$| `dj_db`|

#  Phát biểu bài toán

Giống như các lab trước, bạn sẽ sử dụng ví dụ minh họa về dự đoán giá nhà. Tập dữ liệu huấn luyện chứa nhiều ví dụ với 4 đặc trưng (diện tích, số phòng ngủ, số tầng và tuổi nhà) được thể hiện trong bảng dưới đây. Lưu ý, trong lab này, đặc trưng Diện tích có đơn vị sqft trong khi các lab trước sử dụng đơn vị 1000 sqft. Tập dữ liệu này lớn hơn lab trước.

Chúng ta muốn xây dựng một mô hình hồi quy tuyến tính sử dụng các giá trị này để có thể dự đoán giá của các căn nhà khác - ví dụ, một căn nhà 1200 sqft, 3 phòng ngủ, 1 tầng, 40 năm tuổi. 

##  Tập dữ liệu: 
| Diện tích (sqft) | Số phòng ngủ  | Số tầng | Tuổi nhà | Giá (nghìn đô la)  |   
| ----------------| ------------------- |----------------- |--------------|----------------------- |  
| 952             | 2                   | 1                | 65           | 271.5                  |  
| 1244            | 3                   | 2                | 64           | 232                    |  
| 1947            | 3                   | 2                | 17           | 509.8                  |  
| ...             | ...                 | ...              | ...          | ...                    |


In [ ]:
# nạp tập dữ liệu
X_train, y_train = load_house_data()
X_features = ['size(sqft)','bedrooms','floors','age']

Hãy xem tập dữ liệu và các đặc trưng của nó bằng cách vẽ đồ thị từng đặc trưng theo giá.

In [ ]:
fig,ax=plt.subplots(1, 4, figsize=(12, 3), sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_train[:,i],y_train)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("Price (1000's)")
plt.show()

Việc vẽ đồ thị từng đặc trưng so với biến mục tiêu (giá) cho ta một số dấu hiệu về đặc trưng nào có ảnh hưởng mạnh nhất đến giá. Ở trên, diện tích tăng cũng làm giá tăng. Số phòng ngủ và số tầng dường như không có ảnh hưởng mạnh đến giá. Nhà mới xây có giá cao hơn nhà cũ.

<a name="toc_15456_5"></a>
## Gradient Descent Với Nhiều Biến
Dưới đây là các phương trình bạn đã xây dựng ở lab trước về gradient descent cho nhiều biến:

$$\begin{align*} \text{lặp lại}&\text{ đến khi hội tụ:} \; \lbrace \newline\;
& w_j := w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \tag{1}  \; & \text{với j = 0..n-1}\newline
&b\ \ := b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b}  \newline \rbrace
\end{align*}$$

trong đó, n là số lượng đặc trưng, các tham số $w_j$, $b$ được cập nhật đồng thời, và  

$$
\begin{align}
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_{j}^{(i)} \tag{2}  \\
\frac{\partial J(\mathbf{w},b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) \tag{3}
\end{align}
$$
* m là số lượng ví dụ huấn luyện trong tập dữ liệu

    
*  $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ là giá trị dự đoán của mô hình, còn $y^{(i)}$ là giá trị mục tiêu


## Tốc độ học (Learning Rate)
<figure>
    <img src="./images/C1_W2_Lab06_learningrate.PNG" style="width:1200px;" >
</figure>
Bài giảng đã thảo luận một số vấn đề liên quan đến việc thiết lập tốc độ học $\alpha$. Tốc độ học kiểm soát độ lớn của việc cập nhật các tham số. Xem phương trình (1) ở trên. Nó được dùng chung cho tất cả các tham số.  

Hãy chạy gradient descent và thử một vài giá trị $\alpha$ trên tập dữ liệu của chúng ta

### $\alpha$ = 9.9e-7

In [ ]:
#đặt alpha = 9.9e-7
_, _, hist = run_gradient_descent(X_train, y_train, 10, alpha = 9.9e-7)

Có vẻ như tốc độ học đang quá cao. Nghiệm không hội tụ. Chi phí (cost) đang *tăng* thay vì giảm. Hãy vẽ đồ thị kết quả:

In [ ]:
plot_cost_i_w(X_train, y_train, hist)

Đồ thị bên phải cho thấy giá trị của một trong các tham số, $w_0$. Ở mỗi vòng lặp, nó vượt quá giá trị tối ưu và kết quả là chi phí lại *tăng* thay vì tiến gần đến giá trị nhỏ nhất. Lưu ý rằng đây không phải là bức tranh hoàn toàn chính xác vì có 4 tham số được điều chỉnh ở mỗi lượt chứ không chỉ một. Đồ thị này chỉ hiển thị $w_0$ trong khi các tham số khác được giữ cố định ở các giá trị vô hại. Trong đồ thị này và các đồ thị sau, bạn có thể nhận thấy đường màu xanh và màu cam hơi lệch nhau.


### $\alpha$ = 9e-7
Hãy thử một giá trị nhỏ hơn một chút và xem điều gì xảy ra.

In [ ]:
#đặt alpha = 9e-7
_,_,hist = run_gradient_descent(X_train, y_train, 10, alpha = 9e-7)

Chi phí giảm dần trong suốt quá trình chạy, cho thấy alpha không quá lớn. 

In [ ]:
plot_cost_i_w(X_train, y_train, hist)

Ở bên trái, bạn thấy chi phí đang giảm như mong đợi. Ở bên phải, bạn có thể thấy $w_0$ vẫn đang dao động quanh giá trị nhỏ nhất, nhưng chi phí giảm dần ở mỗi vòng lặp thay vì tăng lên. Lưu ý ở trên rằng `dj_dw[0]` đổi dấu ở mỗi vòng lặp khi `w[0]` nhảy qua giá trị tối ưu.
Giá trị alpha này sẽ hội tụ. Bạn có thể thay đổi số vòng lặp để xem nó hoạt động như thế nào.

### $\alpha$ = 1e-7
Hãy thử một giá trị $\alpha$ nhỏ hơn một chút và xem điều gì xảy ra.

In [ ]:
#đặt alpha = 1e-7
_,_,hist = run_gradient_descent(X_train, y_train, 10, alpha = 1e-7)

Chi phí giảm dần trong suốt quá trình chạy, cho thấy $\alpha$ không quá lớn. 

In [ ]:
plot_cost_i_w(X_train,y_train,hist)

Ở bên trái, bạn thấy chi phí đang giảm như mong đợi. Ở bên phải, bạn có thể thấy $w_0$ đang tiến gần đến giá trị nhỏ nhất mà không dao động. `dj_w0` âm trong suốt quá trình chạy. Nghiệm này cũng sẽ hội tụ.

## Chuẩn hóa đặc trưng (Feature Scaling)
<figure>
    <img src="./images/C1_W2_Lab06_featurescalingheader.PNG" style="width:1200px;" >
</figure>
Bài giảng đã mô tả tầm quan trọng của việc chuẩn hóa lại tập dữ liệu để các đặc trưng có phạm vi giá trị tương tự nhau.
Nếu bạn quan tâm đến chi tiết vì sao lại như vậy, hãy nhấp vào tiêu đề 'Chi tiết' bên dưới. Nếu không, phần dưới đây sẽ hướng dẫn cách triển khai việc chuẩn hóa đặc trưng.

<details>
<summary>
    <font size='3', color='darkgreen'><b>Chi tiết</b></font>
</summary>

Hãy nhìn lại tình huống với $\alpha$ = 9e-7. Đây là giá trị khá gần với giá trị lớn nhất mà chúng ta có thể đặt cho $\alpha$ mà không bị phân kỳ. Đây là một lượt chạy ngắn cho thấy vài vòng lặp đầu tiên:

<figure>
    <img src="./images/C1_W2_Lab06_ShortRun.PNG" style="width:1200px;" >
</figure>

Ở trên, trong khi chi phí đang giảm, có thể thấy rõ $w_0$ tiến triển nhanh hơn nhiều so với các tham số khác do đạo hàm (gradient) của nó lớn hơn nhiều.

Hình bên dưới cho thấy kết quả của một lượt chạy rất dài với $\alpha$ = 9e-7. Việc này mất vài giờ.

<figure>
    <img src="./images/C1_W2_Lab06_LongRun.PNG" style="width:1200px;" >
</figure>
    
Ở trên, bạn có thể thấy chi phí giảm chậm lại sau khi giảm mạnh ban đầu. Hãy chú ý sự khác biệt giữa `w0` và `w1`,`w2`,`w3` cũng như giữa `dj_dw0` và `dj_dw1-3`. `w0` đạt gần đến giá trị cuối cùng rất nhanh và `dj_dw0` đã nhanh chóng giảm xuống giá trị nhỏ, cho thấy `w0` đang gần với giá trị cuối cùng. Các tham số khác giảm chậm hơn nhiều.

Tại sao lại như vậy? Liệu có điều gì chúng ta có thể cải thiện không? Xem bên dưới:
<figure>
    <center> <img src="./images/C1_W2_Lab06_scale.PNG"   ></center>
</figure>   

Hình trên cho thấy vì sao các $w$ được cập nhật không đồng đều. 
- $\alpha$ được dùng chung cho tất cả các lần cập nhật tham số ($w$'s và $b$).
- số hạng lỗi (error term) chung được nhân với các đặc trưng cho các $w$'s. (không phải cho $b$).
- các đặc trưng có độ lớn khác nhau đáng kể khiến một số đặc trưng cập nhật nhanh hơn nhiều so với các đặc trưng khác. Trong trường hợp này, $w_0$ được nhân với 'diện tích (sqft)', thường > 1000, trong khi $w_1$ được nhân với 'số phòng ngủ', thường là 2-4. 
    
Giải pháp là Chuẩn hóa đặc trưng (Feature Scaling).

Bài giảng đã thảo luận ba kỹ thuật khác nhau: 
- Chuẩn hóa đặc trưng (Feature scaling), về cơ bản là chia mỗi đặc trưng dương cho giá trị lớn nhất của nó, hoặc tổng quát hơn, chuẩn hóa lại mỗi đặc trưng bằng cả giá trị nhỏ nhất và lớn nhất theo công thức (x-min)/(max-min). Cả hai cách đều chuẩn hóa các đặc trưng về phạm vi -1 đến 1, trong đó cách đầu tiên phù hợp với các đặc trưng dương, đơn giản và phù hợp với ví dụ trong bài giảng, còn cách thứ hai phù hợp với mọi loại đặc trưng.
- Chuẩn hóa trung bình (Mean normalization): $x_i := \dfrac{x_i - \mu_i}{max - min} $ 
- Chuẩn hóa Z-score, kỹ thuật chúng ta sẽ tìm hiểu bên dưới. 


### chuẩn hóa z-score 
Sau khi chuẩn hóa z-score, tất cả các đặc trưng sẽ có trung bình là 0 và độ lệch chuẩn là 1.

Để thực hiện chuẩn hóa z-score, hãy điều chỉnh các giá trị đầu vào theo công thức sau:
$$x^{(i)}_j = \dfrac{x^{(i)}_j - \mu_j}{\sigma_j} \tag{4}$$ 
trong đó $j$ chọn một đặc trưng hay một cột trong ma trận $\mathbf{X}$. $µ_j$ là giá trị trung bình của tất cả các giá trị của đặc trưng (j) và $\sigma_j$ là độ lệch chuẩn của đặc trưng (j).
$$
\begin{align}
\mu_j &= \frac{1}{m} \sum_{i=0}^{m-1} x^{(i)}_j \tag{5}\\
\sigma^2_j &= \frac{1}{m} \sum_{i=0}^{m-1} (x^{(i)}_j - \mu_j)^2  \tag{6}
\end{align}
$$

>**Lưu ý khi triển khai:** Khi chuẩn hóa các đặc trưng, điều quan trọng
là phải lưu lại các giá trị được dùng để chuẩn hóa - giá trị trung bình và độ lệch chuẩn được dùng trong quá trình tính toán. Sau khi học được các tham số
của mô hình, chúng ta thường muốn dự đoán giá của những căn nhà chưa
từng thấy trước đây. Với một giá trị x mới (diện tích phòng khách và số phòng
ngủ), trước tiên ta phải chuẩn hóa x bằng giá trị trung bình và độ lệch chuẩn
đã tính trước đó từ tập huấn luyện.

**Triển khai**

In [ ]:
def zscore_normalize_features(X):
    """
    computes  X, zcore normalized by column
    
    Args:
      X (ndarray (m,n))     : input data, m examples, n features
      
    Returns:
      X_norm (ndarray (m,n)): input normalized by column
      mu (ndarray (n,))     : mean of each feature
      sigma (ndarray (n,))  : standard deviation of each feature
    """
    # find the mean of each column/feature
    mu     = np.mean(X, axis=0)                 # mu will have shape (n,)
    # find the standard deviation of each column/feature
    sigma  = np.std(X, axis=0)                  # sigma will have shape (n,)
    # element-wise, subtract mu for that column from each example, divide by std for that column
    X_norm = (X - mu) / sigma      

    return (X_norm, mu, sigma)
 
#check our work
#from sklearn.preprocessing import scale
#scale(X_orig, axis=0, with_mean=True, with_std=True, copy=True)

Hãy xem các bước liên quan đến chuẩn hóa Z-score. Đồ thị dưới đây cho thấy từng bước biến đổi.

In [ ]:
mu     = np.mean(X_train,axis=0)   
sigma  = np.std(X_train,axis=0) 
X_mean = (X_train - mu)
X_norm = (X_train - mu)/sigma      

fig,ax=plt.subplots(1, 3, figsize=(12, 3))
ax[0].scatter(X_train[:,0], X_train[:,3])
ax[0].set_xlabel(X_features[0]); ax[0].set_ylabel(X_features[3]);
ax[0].set_title("unnormalized")
ax[0].axis('equal')

ax[1].scatter(X_mean[:,0], X_mean[:,3])
ax[1].set_xlabel(X_features[0]); ax[0].set_ylabel(X_features[3]);
ax[1].set_title(r"X - $\mu$")
ax[1].axis('equal')

ax[2].scatter(X_norm[:,0], X_norm[:,3])
ax[2].set_xlabel(X_features[0]); ax[0].set_ylabel(X_features[3]);
ax[2].set_title(r"Z-score normalized")
ax[2].axis('equal')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
fig.suptitle("distribution of features before, during, after normalization")
plt.show()

Đồ thị trên cho thấy mối quan hệ giữa hai tham số của tập huấn luyện, "tuổi nhà" và "diện tích (sqft)". *Chúng được vẽ với cùng một tỉ lệ*. 
- Trái: Chưa chuẩn hóa: Phạm vi giá trị hay phương sai của đặc trưng 'diện tích (sqft)' lớn hơn nhiều so với 'tuổi nhà'
- Giữa: Bước đầu tiên loại bỏ giá trị trung bình khỏi mỗi đặc trưng. Điều này khiến các đặc trưng tập trung quanh giá trị 0. Khó thấy sự khác biệt đối với đặc trưng 'tuổi nhà', nhưng 'diện tích (sqft)' thì rõ ràng tập trung quanh 0.
- Phải: Bước thứ hai chia cho độ lệch chuẩn. Điều này khiến cả hai đặc trưng đều tập trung quanh 0 với tỉ lệ tương tự nhau.

Hãy chuẩn hóa dữ liệu và so sánh nó với dữ liệu gốc.

In [ ]:
# chuẩn hóa các đặc trưng gốc
X_norm, X_mu, X_sigma = zscore_normalize_features(X_train)
print(f"X_mu = {X_mu}, \nX_sigma = {X_sigma}")
print(f"Peak to Peak range by column in Raw        X:{np.ptp(X_train,axis=0)}")   
print(f"Peak to Peak range by column in Normalized X:{np.ptp(X_norm,axis=0)}")

Phạm vi giá trị lớn nhất - nhỏ nhất (peak to peak) của mỗi cột đã giảm từ mức hàng nghìn xuống chỉ còn hệ số 2-3 nhờ chuẩn hóa.

In [ ]:
fig,ax=plt.subplots(1, 4, figsize=(12, 3))
for i in range(len(ax)):
    norm_plot(ax[i],X_train[:,i],)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("count");
fig.suptitle("distribution of features before normalization")
plt.show()
fig,ax=plt.subplots(1,4,figsize=(12,3))
for i in range(len(ax)):
    norm_plot(ax[i],X_norm[:,i],)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("count"); 
fig.suptitle("distribution of features after normalization")

plt.show()

Lưu ý ở trên, phạm vi của dữ liệu đã chuẩn hóa (trục x) tập trung quanh 0 và khoảng +/- 2. Quan trọng nhất, phạm vi này tương tự nhau đối với mỗi đặc trưng.

Hãy chạy lại thuật toán gradient descent với dữ liệu đã chuẩn hóa.
Lưu ý **giá trị alpha lớn hơn rất nhiều**. Điều này sẽ giúp gradient descent chạy nhanh hơn.

In [ ]:
w_norm, b_norm, hist = run_gradient_descent(X_norm, y_train, 1000, 1.0e-1, )

Các đặc trưng đã được chuẩn hóa cho kết quả rất chính xác **nhanh hơn rất nhiều!**. Lưu ý đạo hàm (gradient) của mỗi tham số rất nhỏ vào cuối lượt chạy khá ngắn này. Tốc độ học 0.1 là một điểm khởi đầu tốt cho hồi quy với các đặc trưng đã chuẩn hóa.
Hãy vẽ đồ thị dự đoán của chúng ta so với giá trị mục tiêu. Lưu ý, việc dự đoán được thực hiện bằng đặc trưng đã chuẩn hóa trong khi đồ thị được hiển thị bằng giá trị đặc trưng gốc.

In [ ]:
#dự đoán giá trị mục tiêu bằng các đặc trưng đã chuẩn hóa
m = X_norm.shape[0]
yp = np.zeros(m)
for i in range(m):
    yp[i] = np.dot(X_norm[i], w_norm) + b_norm

    # vẽ dự đoán và giá trị mục tiêu theo các đặc trưng gốc    
fig,ax=plt.subplots(1,4,figsize=(12, 3),sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_train[:,i],y_train, label = 'target')
    ax[i].set_xlabel(X_features[i])
    ax[i].scatter(X_train[:,i],yp,color=dlc["dlorange"], label = 'predict')
ax[0].set_ylabel("Price"); ax[0].legend();
fig.suptitle("target versus prediction using z-score normalized model")
plt.show()

Kết quả trông khá tốt. Một vài điểm cần lưu ý:
- với nhiều đặc trưng, chúng ta không thể chỉ dùng một đồ thị duy nhất để hiển thị kết quả theo các đặc trưng.
- khi vẽ đồ thị, các đặc trưng đã chuẩn hóa được sử dụng. Bất kỳ dự đoán nào sử dụng các tham số học được từ tập huấn luyện đã chuẩn hóa cũng phải được chuẩn hóa.

**Dự đoán**
Mục đích của việc xây dựng mô hình là để sử dụng nó dự đoán giá nhà không có trong tập dữ liệu. Hãy dự đoán giá của một căn nhà có 1200 sqft, 3 phòng ngủ, 1 tầng, 40 năm tuổi. Hãy nhớ rằng bạn phải chuẩn hóa dữ liệu bằng giá trị trung bình và độ lệch chuẩn đã tính được khi chuẩn hóa dữ liệu huấn luyện. 

In [ ]:
# Đầu tiên, chuẩn hóa ví dụ của chúng ta.
x_house = np.array([1200, 3, 1, 40])
x_house_norm = (x_house - X_mu) / X_sigma
print(x_house_norm)
x_house_predict = np.dot(x_house_norm, w_norm) + b_norm
print(f" predicted price of a house with 1200 sqft, 3 bedrooms, 1 floor, 40 years old = ${x_house_predict*1000:0.0f}")

**Đường đồng mức chi phí (Cost Contours)**  
<img align="left" src="./images/C1_W2_Lab06_contours.PNG"   style="width:240px;" >Một cách khác để xem xét việc chuẩn hóa đặc trưng là thông qua các đường đồng mức chi phí. Khi tỉ lệ của các đặc trưng không khớp nhau, đồ thị đường đồng mức của chi phí theo tham số sẽ bị lệch (bất đối xứng). 

Trong đồ thị bên dưới, tỉ lệ của các tham số được điều chỉnh khớp nhau. Đồ thị bên trái là đường đồng mức chi phí của w[0] (diện tích tính bằng feet vuông) so với w[1] (số phòng ngủ) trước khi chuẩn hóa các đặc trưng. Đồ thị bị lệch đến mức các đường cong hoàn thiện đường đồng mức không thể nhìn thấy được. Ngược lại, khi các đặc trưng đã được chuẩn hóa, đường đồng mức chi phí đối xứng hơn nhiều. Kết quả là việc cập nhật các tham số trong quá trình gradient descent có thể tiến triển đều nhau cho mỗi tham số. 


In [ ]:
plt_equal_scale(X_train, X_norm, y_train)


## Chúc mừng!
Trong lab này, bạn đã:
- sử dụng các hàm hồi quy tuyến tính với nhiều đặc trưng mà bạn đã xây dựng ở các lab trước
- khám phá ảnh hưởng của tốc độ học $\alpha$ đến sự hội tụ 
- khám phá giá trị của việc chuẩn hóa đặc trưng bằng z-score trong việc tăng tốc độ hội tụ

## Lời cảm ơn
Dữ liệu nhà ở được lấy từ [tập dữ liệu Ames Housing](http://jse.amstat.org/v19n3/decock.pdf) do Dean De Cock biên soạn để sử dụng trong giáo dục khoa học dữ liệu.